# The Market Model: Estimating and Understanding Beta

**From CAPM theory to real market data**

In the slides, we learned why beta matters for asset pricing. Here we estimate beta from observed stock and market returns, interpret the results, and investigate how stable those estimates are.

**Key question:** *How sensitive is a stock to market movements, and how reliable is our estimate?*

## 1. The Market Model: A Quick Reminder

$$R_{i,t}=\alpha_i+\beta_iR_{M,t}+\varepsilon_{i,t}$$

- **Beta ($\beta_i$):** how sensitive a stock's return is to the market return.
- **Alpha ($\alpha_i$):** the regression intercept, not automatically evidence of abnormal performance.
- **Residual ($\varepsilon_{i,t}$):** movements not explained by the market regression.

We will use **daily or monthly logarithmic returns** for the empirical regressions. The CAPM is an equilibrium statement about expected *simple* returns; the log-return regressions here estimate market exposure as an empirical approximation.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import yfinance as yf

## 2. Estimating Beta: Two Equivalent Approaches

**Question:** How closely does Endesa move with the IBEX 35?

Download adjusted closing prices, calculate log returns, and estimate the relationship between Endesa and the market. We will first calculate beta from covariances and then estimate an OLS regression.

*Data note:* Yahoo's adjusted Endesa price includes available corporate-action adjustments, whereas the IBEX 35 ticker is generally a price index rather than a total-return index. This matters especially for interpreting long-run alpha.

In [ ]:
# Endesa and IBEX 35, using common trading dates
prices_es = yf.download(
    ['ELE.MC', '^IBEX'], start='2010-01-01', end='2026-01-01',
    auto_adjust=True, progress=False
)['Close'].dropna()

returns_es = np.log(prices_es).diff().dropna()

# Compare normalized price paths
(100 * prices_es / prices_es.iloc[0]).plot(figsize=(10, 4))
plt.title('Endesa and IBEX 35 (initial value = 100)')
plt.ylabel('Normalized price')
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# Beta from covariance and market variance
beta_cov = returns_es['ELE.MC'].cov(returns_es['^IBEX']) / returns_es['^IBEX'].var()

# Beta from the slope of an OLS regression
X = sm.add_constant(returns_es['^IBEX'])
model = sm.OLS(returns_es['ELE.MC'], X).fit()

print(f'Beta (covariance): {beta_cov:.3f}')
print(f'Beta (OLS):        {model.params["^IBEX"]:.3f}')
print(f'Alpha (daily):     {model.params["const"]:.5f}')
print(f'R-squared:         {model.rsquared:.2%}')

In [ ]:
# Plot the observed returns and fitted regression line
plt.figure(figsize=(8, 5))
plt.scatter(returns_es['^IBEX'], returns_es['ELE.MC'], alpha=0.25, s=10)
plt.plot(returns_es['^IBEX'].sort_values(),
         model.predict(sm.add_constant(returns_es['^IBEX'].sort_values())),
         color='red', linewidth=2)
plt.xlabel('IBEX 35 daily log return')
plt.ylabel('Endesa daily log return')
plt.title('Endesa: Estimated Market Model')
plt.grid(alpha=0.3)
plt.show()

## 3. Understanding the Regression

Use `model.summary()` to interpret **alpha**, **beta**, **$R^2$**, and the uncertainty of the beta estimate.

**Think about it:** If beta is 0.8, does Endesa always move 0.8% when the IBEX rises by 1%? Why not? Does a positive intercept prove that the stock is undervalued?

In [ ]:
# Focus on coefficient estimates, confidence intervals and model fit
print(model.summary())
print('95% confidence interval for beta:')
print(model.conf_int().loc['^IBEX'])

## 4. Systematic and Idiosyncratic Risk

In a one-factor OLS regression with an intercept, the fitted and residual components are orthogonal in the sample:

$$\operatorname{Var}(R_i)=\beta_i^2\operatorname{Var}(R_M)+\operatorname{Var}(\varepsilon_i).$$

How much of Endesa's return variance is associated with the market? How much remains unexplained?

In [ ]:
# Decompose Endesa's daily return variance
systematic_var = model.params['^IBEX']**2 * returns_es['^IBEX'].var()
specific_var = model.resid.var()
total_var = returns_es['ELE.MC'].var()

risk_parts = pd.Series({
    'Systematic': systematic_var / total_var,
    'Idiosyncratic': specific_var / total_var
})

risk_parts.plot.bar(figsize=(7, 4), rot=0)
plt.ylabel('Share of total variance')
plt.title('Endesa: Decomposition of Return Variance')
plt.grid(axis='y', alpha=0.3)
plt.show()

print(f'Systematic share: {risk_parts["Systematic"]:.1%}')
print(f'R-squared:        {model.rsquared:.1%}')

## 5. Comparing Stocks: Are Their Betas Similar?

**Exercise:** Compare Amazon, Coca-Cola and Microsoft with the S&P 500. Estimate beta, monthly alpha and $R^2$ for each stock using the same period and market index.

We solve the first regression using the same method as for Endesa; then use a short loop to summarize the three stocks.

In [ ]:
# US stocks and the S&P 500 index
stocks = ['AMZN', 'KO', 'MSFT']
market = '^GSPC'

prices_us = yf.download(
    stocks + [market], start='2010-01-01', end='2026-01-01',
    auto_adjust=True, progress=False
)['Close'].dropna()

# Monthly logarithmic returns (end-of-month prices)
monthly_us = np.log(prices_us.resample('ME').last()).diff().dropna()
monthly_us.head()

In [ ]:
# OLS parameters for each stock, using the same market benchmark
results = []

for stock in stocks:
    regression = sm.OLS(monthly_us[stock], sm.add_constant(monthly_us[market])).fit()
    results.append([stock, regression.params['const'],
                    regression.params[market], regression.rsquared])

stock_results = pd.DataFrame(results, columns=['Asset', 'Monthly Alpha', 'Beta', 'R-squared'])
stock_results = stock_results.set_index('Asset')
display(stock_results.round(3))

In [ ]:
# Compare betas and the proportion of variance explained
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
stock_results['Beta'].plot.bar(ax=axes[0], title='Market Beta', rot=0)
axes[0].axhline(1, linestyle='--', color='gray')
stock_results['R-squared'].plot.bar(ax=axes[1], title='R-squared', rot=0)
for ax in axes:
    ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Rolling Betas: Is Market Exposure Stable?

A single regression gives **one historical beta**. But a company's exposure to market movements may change.

Estimate a new beta each month using the **previous 60 monthly observations**. Plot the three stocks and discuss which estimates change most over time.

In [ ]:
# Rolling 60-month beta = rolling covariance / rolling market variance
window = 60
market_var = monthly_us[market].rolling(window).var()

rolling_betas = pd.DataFrame(index=monthly_us.index)
for stock in stocks:
    rolling_betas[stock] = (
        monthly_us[stock].rolling(window).cov(monthly_us[market]) / market_var
    )

rolling_betas.plot(figsize=(11, 5), linewidth=1.6)
plt.axhline(1, linestyle='--', color='gray', label='Market beta = 1')
plt.title('Rolling 60-Month Market Betas')
plt.xlabel('Date')
plt.ylabel('Beta')
plt.grid(alpha=0.3)
plt.legend()
plt.show()

## 7. Sector Betas: A Student Challenge

**Exercise:** Use the same rolling-beta method to compare three US sector ETFs:

- **XLI:** Industrials
- **ITA:** Aerospace and defense
- **XLE:** Energy

Use the S&P 500 as the benchmark and a 60-month rolling window. Which sector has the highest beta? Which beta varies most over time?

In [ ]:
# Download sector ETFs and the market index
sectors = ['XLI', 'ITA', 'XLE']
prices_sector = yf.download(
    sectors + [market], start='2010-01-01', end='2026-01-01',
    auto_adjust=True, progress=False
)['Close'].dropna()

monthly_sector = np.log(prices_sector.resample('ME').last()).diff().dropna()

# Student task: complete the rolling-beta calculation for each ETF
# Hint: reproduce the formula used in Section 6 with monthly_sector.

## 8. Does Return Frequency Change Estimated Beta?

**Exercise:** For Endesa and the IBEX 35, compare beta estimated from **daily, weekly and monthly log returns**. Use the same underlying price history for all three regressions.

Why might the estimates differ? Remember: log returns can be aggregated over time, but beta is estimated from the covariance structure at the selected frequency.

In [ ]:
# Create returns at three frequencies
frequency_returns = {
    'Daily': np.log(prices_es).diff().dropna(),
    'Weekly': np.log(prices_es.resample('W-FRI').last()).diff().dropna(),
    'Monthly': np.log(prices_es.resample('ME').last()).diff().dropna()
}

# Estimate beta consistently at each frequency
frequency_beta = {}
for frequency, data in frequency_returns.items():
    frequency_beta[frequency] = data['ELE.MC'].cov(data['^IBEX']) / data['^IBEX'].var()

pd.Series(frequency_beta, name='Endesa Beta').to_frame().round(3)

## 9. From Estimated Beta to CAPM Expected Return

The **Market Model** estimates beta from historical co-movements. The **CAPM** uses beta to relate an asset's *expected simple return* to its systematic risk:

$$E[R_i]=r_f+\beta_i\bigl(E[R_M]-r_f\bigr).$$

**Final exercise:** Suppose the annual risk-free rate is 3% and the expected annual market premium is 6%. Use the estimated betas of AMZN, KO and MSFT as proxies to calculate the CAPM-implied expected annual simple returns.

Do these predictions have to match the stocks' historical average returns? Explain why not.

In [ ]:
# Illustrative CAPM expected annual simple returns
rf_annual = 0.03
market_premium = 0.06

capm_returns = rf_annual + stock_results['Beta'] * market_premium

capm_table = pd.DataFrame({
    'Estimated Beta': stock_results['Beta'],
    'CAPM Expected Return': capm_returns
})

display(capm_table.style.format({
    'Estimated Beta': '{:.2f}',
    'CAPM Expected Return': '{:.2%}'
}))

## Takeaways

- **Beta** measures how stock returns co-move with the market; it can be estimated by covariance or an OLS slope.
- **$R^2$** describes how much sample return variance the market regression explains.
- Beta can differ across stocks, sectors, estimation windows and return frequencies.
- A beta estimate is an input to CAPM; **it does not guarantee a future return**.

**Further reading:** Sharpe (1964), *The Journal of Finance*; Fama and MacBeth (1973), *Journal of Political Economy*.